# ComfyUI en Kaggle (2× T4) — servidor API v5 (base: v4)

**v5 = v4 + modo servidor:** bucle final que mantiene vivo ComfyUI y el túnel (y los reinicia si caen), cabeceras de Cloudflare Access opcionales y cliente para PC en `comfy-kaggle-stack/client/comfy_remote.py`.

**Cómo usarlo**
1. Settings del notebook: **Accelerator = GPU T4 x2**, **Internet = On**.
2. (Opcional) *Add-ons → Secrets*: `HF_TOKEN` (Hugging Face) y `CF_TUNNEL_TOKEN` (túnel Cloudflare con nombre).
3. Ejecuta **en orden**. Cada fase tiene un criterio de éxito. Si una fase falla, **no avances**: anota el resultado en la Bitácora del documento.
4. Las celdas son re-ejecutables: si algo ya existe, se salta.
5. **Reglas aprendidas (v4):** los modelos se montan desde Datasets de Kaggle con `kagglehub` (lista `KAGGLE_DATASETS`, sin descarga ni disco); lo que falte se baja de Hugging Face; abre el túnel en ventana de incógnito si el navegador da 403; **nunca** descargues modelos desde las plantillas de la interfaz (escriben en /kaggle/working, límite 19.5 GiB); los modelos van a /tmp y se borran al cerrar la sesión.

| Fase | Qué valida | Éxito |
|---|---|---|
| 0 | Entorno real (GPU, sm, bf16, RAM, disco) | Confirma sm_75, 2 GPUs, disco suficiente |
| 1 | ComfyUI + nodos + servidor + túnel | `/system_stats` responde |
| 2 | Imagen: Z-Image Turbo en fp16 | Imagen no negra, < ~90 s |
| 2B | Plan B imagen: SDXL + Lightning | Solo si falla la fase 2 |
| 3 | Video: Wan 2.2 I2V A14B GGUF Q4 + lightx2v | 33 frames OK → luego 81 |
| 4 | API para n8n | Workflows JSON exportados |

## Configuración
Único lugar donde cambias parámetros.

In [ ]:
# ===== CONFIG =====
COMFY_DIR   = "/kaggle/working/ComfyUI"   # si ya tienes ComfyUI en otra ruta, cámbiala aquí
MODELS_DIR  = None                         # None = autodetecta el disco con más espacio libre
OUT_DIR     = "/kaggle/working/outputs"
LOG_PATH    = "/kaggle/working/bitacora.jsonl"
PORT_A      = 8188     # instancia principal: ve las 2 GPUs (difusión en cuda:0, encoders/VAE en cuda:1)
PORT_B      = 8189     # instancia secundaria opcional: solo GPU 1 (lotes de imagen en paralelo)
RUN_SECOND_INSTANCE = False
USE_KAGGLE_INPUT    = True   # busca primero los modelos en /kaggle/input; si no están, descarga de Hugging Face
# Datasets de Kaggle que se montan con kagglehub (sin descarga, sin ocupar disco, sin "Add Input" manual).
# Cuando crees tu propio dataset (celda 1.2C), pon su handle PRIMERO, p.ej. "tu_usuario/comfyui-t4-models".
KAGGLE_DATASETS = [
    "nguynxuncngde180528/wan22-i2v-a14b-gguf",   # Wan 2.2 I2V HighNoise/LowNoise Q4_K_M + VAE (mismos bytes que QuantStack)
    "rulex8/umt5-xxl-fp16",                      # text encoder UMT5-XXL fp16
]
ENABLE_TUNNEL       = True
COMFY_EXTRA_ARGS    = []   # ajustes finos más adelante, p.ej. ["--fp32-vae"], ["--reserve-vram","1"]

import os
os.makedirs(OUT_DIR, exist_ok=True)

def get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)

HF_TOKEN        = get_secret("HF_TOKEN")
CF_TUNNEL_TOKEN = get_secret("CF_TUNNEL_TOKEN")
print("HF_TOKEN:", "sí" if HF_TOKEN else "no", "| CF_TUNNEL_TOKEN:", "sí" if CF_TUNNEL_TOKEN else "no (se usará túnel rápido)")
# ---- v5: modo servidor y Cloudflare Access (opcional) ----
SERVER_MODE     = True     # True = la última celda deja el notebook vivo vigilando ComfyUI y el túnel
SERVER_HOURS    = 11       # tope de seguridad (Kaggle corta las sesiones; 12 h es el máximo)
CF_ACCESS_ID    = get_secret("CF_ACCESS_CLIENT_ID")      # service token de Cloudflare Access (opcional)
CF_ACCESS_SECRET= get_secret("CF_ACCESS_CLIENT_SECRET")
ACCESS_HEADERS  = {"CF-Access-Client-Id": CF_ACCESS_ID, "CF-Access-Client-Secret": CF_ACCESS_SECRET} if CF_ACCESS_ID and CF_ACCESS_SECRET else {}
print("Cloudflare Access:", "service token cargado" if ACCESS_HEADERS else "sin token (el túnel quedaría abierto)")


## Fase 0 — Diagnóstico del entorno (solo lectura)
No instala ni cambia nada. Mide lo que el hardware es realmente.

In [ ]:
import subprocess, shutil, json, platform, sys, time
import torch, psutil

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

print(sh("nvidia-smi --query-gpu=index,name,memory.total,memory.used,temperature.gpu,power.limit --format=csv"))
print("\nPython", platform.python_version(), "| torch", torch.__version__, "| CUDA", torch.version.cuda)

diag = {"gpus": []}
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    g = {"idx": i, "name": p.name, "sm": f"{p.major}{p.minor}", "vram_gib": round(p.total_memory / 2**30, 2),
         "bf16_nativo": p.major >= 8, "fp8_nativo": (p.major, p.minor) >= (8, 9)}
    diag["gpus"].append(g); print(g)

try:
    import xformers; diag["xformers"] = xformers.__version__
except Exception as e:
    diag["xformers"] = f"no disponible ({e})"
print("xformers:", diag["xformers"])

vm = psutil.virtual_memory()
diag["ram_gib"] = round(vm.total / 2**30, 1)
print(f"RAM total {diag['ram_gib']} GiB | libre {vm.available/2**30:.1f} GiB")

diag["disks"] = {}
for path in ["/kaggle/working", "/kaggle/tmp", "/tmp", "/root"]:
    if os.path.exists(path):
        du = shutil.disk_usage(path)
        diag["disks"][path] = round(du.free / 2**30, 1)
        print(f"Disco {path:16s} libre {du.free/2**30:7.1f} GiB / total {du.total/2**30:7.1f} GiB")

### Micro-benchmark fp16 vs bf16
Demuestra en tu hardware por qué los modelos corren en fp16: en Turing bf16 se emula y es mucho más lento.

In [ ]:
def bench(dtype, n=4096, it=20, dev="cuda:0"):
    a = torch.randn(n, n, device=dev, dtype=dtype); b = torch.randn(n, n, device=dev, dtype=dtype)
    for _ in range(3): a @ b
    torch.cuda.synchronize(); t = time.time()
    for _ in range(it): a @ b
    torch.cuda.synchronize()
    return 2 * n**3 * it / (time.time() - t) / 1e12

for dt in [torch.float16, torch.bfloat16, torch.float32]:
    try:
        print(f"{str(dt):15s} {bench(dt):6.1f} TFLOPS")
    except Exception as e:
        print(dt, "ERROR", e)
torch.cuda.empty_cache()
diag["ts"] = time.strftime("%Y-%m-%d %H:%M")
json.dump(diag, open("/kaggle/working/diag.json", "w"), indent=2)
print("\nGuardado en /kaggle/working/diag.json  -> pega el resumen en la Bitácora del documento")

## Fase 1 — ComfyUI, nodos, servidor y túnel

In [ ]:
# 1.1 ComfyUI (solo si no existe) + nodos personalizados mínimos
if not os.path.exists(COMFY_DIR):
    print(sh(f"git clone --depth 1 https://github.com/Comfy-Org/ComfyUI {COMFY_DIR}"))
    print(sh(f"cd {COMFY_DIR} && pip install -q -r requirements.txt")[-2000:])
else:
    print("ComfyUI ya existe:", COMFY_DIR)

CUSTOM_NODES = {
    "ComfyUI-GGUF":              "https://github.com/city96/ComfyUI-GGUF",            # carga modelos .gguf (Wan 14B)
    "ComfyUI-MultiGPU":          "https://github.com/pollockjj/ComfyUI-MultiGPU",     # encoders/VAE en cuda:1
    "ComfyUI-VideoHelperSuite":  "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite",  # guardar video (respaldo)
}
for name, url in CUSTOM_NODES.items():
    d = f"{COMFY_DIR}/custom_nodes/{name}"
    if not os.path.exists(d):
        print(sh(f"git clone --depth 1 {url} {d}")[-300:])
        if os.path.exists(f"{d}/requirements.txt"):
            print(sh(f"pip install -q -r {d}/requirements.txt")[-500:])
    print("OK", name)
print(sh("pip install -q gguf huggingface_hub")[-300:])
print("torch sigue en:", sh(f"{sys.executable} -c 'import torch;print(torch.__version__)'"))

In [ ]:
# 1.2 Carpeta de modelos: el disco con más espacio libre (los modelos se enlazan dentro de ComfyUI/models)
if MODELS_DIR is None:
    cands = {p: shutil.disk_usage(p).free for p in ["/kaggle/tmp", "/tmp", "/kaggle/working"] if os.path.exists(p)}
    best = max(cands, key=cands.get)
    MODELS_DIR = f"{best}/models_store"
os.makedirs(MODELS_DIR, exist_ok=True)
print("MODELS_DIR =", MODELS_DIR, f"| libre {shutil.disk_usage(MODELS_DIR).free/2**30:.1f} GiB")

from fnmatch import fnmatch
from huggingface_hub import list_repo_files, hf_hub_download

# Montar datasets con kagglehub: dentro de Kaggle NO descarga; monta en /kaggle/input desde la caché compartida de Kaggle
print(sh("pip install -q kagglehub 2>&1 | tail -1"))
import kagglehub
MOUNTED = {}
if USE_KAGGLE_INPUT:
    for h in KAGGLE_DATASETS:
        t0 = time.time()
        try:
            MOUNTED[h] = kagglehub.dataset_download(h)
            print(f"  montado {h} -> {MOUNTED[h]} en {time.time()-t0:.0f} s")
        except Exception as e:
            print(f"  ! no se pudo montar {h}: {e}")

_INPUT_INDEX = None
def _input_index():
    """Lista (una vez) todos los archivos de /kaggle/input: datasets y modelos adjuntos con 'Add Input'."""
    global _INPUT_INDEX
    if _INPUT_INDEX is None:
        _INPUT_INDEX = []
        if USE_KAGGLE_INPUT and os.path.isdir("/kaggle/input"):
            for root, _, files in os.walk("/kaggle/input"):
                _INPUT_INDEX += [os.path.join(root, f) for f in files]
    return _INPUT_INDEX

def _link(src, folder, base):
    link = f"{COMFY_DIR}/models/{folder}/{base}"
    os.makedirs(os.path.dirname(link), exist_ok=True)
    if os.path.islink(link) and not os.path.exists(link):
        os.remove(link)   # enlace roto de una sesión anterior (/tmp se borra al cerrar)
    if not os.path.exists(link):
        os.symlink(src, link)
    return link

def fetch(candidates, folder):
    """candidates = [(repo, patrón_glob), ...] en orden de preferencia. Devuelve el nombre de archivo para ComfyUI.
    Orden: 1) ya enlazado y válido  2) /kaggle/input (sin descarga)  3) Hugging Face -> MODELS_DIR."""
    # 2) /kaggle/input: compara el nombre de archivo (sin carpeta, sin mayúsculas)
    for repo, pattern in candidates:
        norm = lambda x: x.lower().replace("_", "").replace("-", "")   # Wan2.1_VAE == wan_2.1_vae
        name_pat = norm(pattern.split("/")[-1])
        hits = sorted(f for f in _input_index() if fnmatch(norm(os.path.basename(f)), name_pat))
        if hits:
            base = os.path.basename(hits[0])
            _link(hits[0], folder, base)
            print(f"  /kaggle/input -> {base} ({os.path.getsize(hits[0])/2**30:.2f} GiB, sin descarga)")
            return base
    # 3) Hugging Face
    for repo, pattern in candidates:
        try:
            files = list_repo_files(repo, token=HF_TOKEN)
        except Exception as e:
            print(f"  ! {repo}: {e}"); continue
        hits = sorted(f for f in files if fnmatch(f, pattern))
        if not hits:
            key = pattern.replace("*", " ").split()[-1][:12]
            print(f"  ! sin coincidencia {pattern} en {repo}. Parecidos:", [f for f in files if key.lower() in f.lower()][:8])
            continue
        f = hits[0]
        base = os.path.basename(f)
        link = f"{COMFY_DIR}/models/{folder}/{base}"
        if os.path.islink(link) and os.path.exists(link):
            print("  ya está:", base); return base
        print(f"  descargando {repo} :: {f}")
        t0 = time.time()
        p = hf_hub_download(repo, f, local_dir=f"{MODELS_DIR}/{repo.replace('/', '__')}", token=HF_TOKEN)
        _link(p, folder, base)
        print(f"  OK {base} en {time.time()-t0:.0f} s | disco libre {shutil.disk_usage(MODELS_DIR).free/2**30:.1f} GiB")
        return base
    raise RuntimeError(f"No se pudo resolver ningún candidato para {folder}: {candidates}")

print("Archivos en /kaggle/input:", len(_input_index()))

### 1.2B — (Opcional) Medir velocidad de lectura de modelos
Compara leer un modelo desde /kaggle/input (dataset adjunto) y desde /tmp (descargado). Ejecutar con ComfyUI apagado o antes de generar, para que la caché no falsee el dato.

In [ ]:
def read_speed(path, mb=4096):
    """Lee los primeros `mb` MB del archivo y devuelve MB/s (lectura en frío si no se leyó antes)."""
    out = sh(f"dd if='{path}' of=/dev/null bs=16M count={mb//16} 2>&1 | tail -1")
    return out
for f in [x for x in _input_index() if x.endswith((".gguf", ".safetensors"))][:2]:
    print("INPUT", os.path.basename(f), "->", read_speed(f))
for f in sh(f"find {MODELS_DIR} -name '*.gguf' -o -name '*.safetensors' | head -2").splitlines():
    print("TMP  ", os.path.basename(f), "->", read_speed(f))

### 1.2C — (Opcional, una sola vez) Crear tu propio Dataset con TODOS los modelos validados
Sube con `kagglehub.dataset_upload` los modelos ya descargados en /tmp (Wan, encoders, VAEs, LoRAs, Z-Image) a un Dataset de tu cuenta. En sesiones siguientes pones su handle primero en `KAGGLE_DATASETS` y se monta sin descargar nada de Hugging Face. Requiere el secret `KAGGLE_API_TOKEN` (o `KAGGLE_USERNAME` + `KAGGLE_KEY`). **No probado aún**: anota tiempo y errores en la bitácora.

In [ ]:
CREATE_DATASET = False                    # cámbialo a True solo cuando quieras subirlo
MY_DATASET     = "TU_USUARIO/comfyui-t4-models"   # handle {usuario}/{nombre}
if CREATE_DATASET:
    for k in ["KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"]:
        v = get_secret(k)
        if v: os.environ[k] = v
    stage = "/tmp/dataset_stage"; os.makedirs(stage, exist_ok=True)
    # Reúne en un solo directorio TODOS los modelos que usa ComfyUI, vengan de HF (/tmp) o de un dataset montado
    for folder in ["diffusion_models", "text_encoders", "vae", "loras", "checkpoints"]:
        d = f"{COMFY_DIR}/models/{folder}"
        if not os.path.isdir(d): continue
        for name in os.listdir(d):
            src, dst = os.path.realpath(f"{d}/{name}"), f"{stage}/{name}"
            if not os.path.isfile(src) or os.path.exists(dst): continue
            if src.startswith("/tmp"):
                os.link(src, dst)            # enlace duro: mismo disco, sin copiar
            elif src.startswith("/kaggle/input"):
                shutil.copy(src, dst)        # montado de solo lectura: hay que copiarlo a /tmp
                print("  copiado", name)
    print(sh(f"ls -la {stage}; du -sh {stage}"))
    t0 = time.time()
    kagglehub.dataset_upload(MY_DATASET, stage, version_notes="Modelos ComfyUI validados en 2x T4")
    print(f"subida terminada en {time.time()-t0:.0f} s -> añade '{MY_DATASET}' al inicio de KAGGLE_DATASETS")

In [ ]:
# 1.3 Arrancar el servidor ComfyUI en segundo plano
import requests

def start_comfy(port, gpu=None, log="/kaggle/working/comfy_{port}.log"):
    # gpu=None -> ve ambas GPUs (necesario para MultiGPU). gpu=1 -> instancia aislada en GPU 1.
    try:
        requests.get(f"http://127.0.0.1:{port}/system_stats", timeout=2); print(f"ya corriendo en :{port}"); return
    except Exception:
        pass
    args = [sys.executable, "main.py", "--listen", "0.0.0.0", "--port", str(port), "--preview-method", "none"] + COMFY_EXTRA_ARGS
    if gpu is not None:
        args += ["--cuda-device", str(gpu)]
    lf = open(log.format(port=port), "w")
    subprocess.Popen(args, cwd=COMFY_DIR, stdout=lf, stderr=subprocess.STDOUT)
    for _ in range(120):
        try:
            s = requests.get(f"http://127.0.0.1:{port}/system_stats", timeout=2).json()
            print(f":{port} listo ->", [(d["name"], round(d["vram_total"]/2**30, 1)) for d in s["devices"]]); return
        except Exception:
            time.sleep(2)
    print(sh(f"tail -40 {log.format(port=port)}")); raise RuntimeError("ComfyUI no arrancó; revisa el log de arriba")

start_comfy(PORT_A)
if RUN_SECOND_INSTANCE:
    start_comfy(PORT_B, gpu=1)

In [ ]:
# 1.4 Túnel Cloudflare
# - Con CF_TUNNEL_TOKEN: túnel con nombre -> hostname fijo (configura en Cloudflare: hostname público -> http://localhost:8188)
# - Sin token: túnel rápido trycloudflare -> URL distinta cada sesión (solo pruebas)
# ComfyUI NO tiene autenticación: en producción pon Cloudflare Access (service token) delante del hostname.
import re
CF = "/kaggle/working/cloudflared"
if ENABLE_TUNNEL:
    if not os.path.exists(CF):
        print(sh(f"wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O {CF} && chmod +x {CF}"))
    cf_log = open("/kaggle/working/cloudflared.log", "w")
    if CF_TUNNEL_TOKEN:
        subprocess.Popen([CF, "tunnel", "--no-autoupdate", "run", "--token", CF_TUNNEL_TOKEN], stdout=cf_log, stderr=subprocess.STDOUT)
        time.sleep(8); print("Túnel con nombre iniciado. Usa tu hostname configurado en Cloudflare.")
    else:
        subprocess.Popen([CF, "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT_A}"], stdout=cf_log, stderr=subprocess.STDOUT)
        url = None
        for _ in range(30):
            time.sleep(2)
            m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/kaggle/working/cloudflared.log").read())
            if m: url = m.group(0); break
        print("URL pública:", url)

## Utilidades de API (usadas por todas las fases y por n8n)

In [ ]:
import uuid, threading
import numpy as np
from PIL import Image
from IPython.display import display, Image as IPImage, Video

BASE = f"http://127.0.0.1:{PORT_A}"
OBJECT_INFO = requests.get(f"{BASE}/object_info").json()

def has_node(n): return n in OBJECT_INFO

def validate(wf):
    missing = sorted({v["class_type"] for v in wf.values() if not has_node(v["class_type"])})
    if missing: raise RuntimeError(f"Nodos no instalados: {missing}")

class VramMonitor(threading.Thread):
    def __init__(self):
        super().__init__(daemon=True); self.peak = {}; self.stop = False
    def run(self):
        while not self.stop:
            for line in sh("nvidia-smi --query-gpu=index,memory.used --format=csv,noheader,nounits").splitlines():
                i, m = [x.strip() for x in line.split(",")]
                self.peak[i] = max(self.peak.get(i, 0), int(m))
            time.sleep(0.5)

def free_memory(base=BASE):
    requests.post(f"{base}/free", json={"unload_models": True, "free_memory": True})
    time.sleep(3)

def upload_image(path, base=BASE):
    with open(path, "rb") as f:
        r = requests.post(f"{base}/upload/image", files={"image": (os.path.basename(path), f)}, data={"overwrite": "true"})
    r.raise_for_status(); return r.json()["name"]

def run_workflow(wf, base=BASE, timeout=7200, tag="run"):
    validate(wf)
    r = requests.post(f"{base}/prompt", json={"prompt": wf, "client_id": str(uuid.uuid4())})
    if r.status_code != 200:
        print(r.text[:3000]); raise RuntimeError("ComfyUI rechazó el workflow")
    pid = r.json()["prompt_id"]
    mon = VramMonitor(); mon.start(); t0 = time.time()
    while True:
        h = requests.get(f"{base}/history/{pid}").json()
        if pid in h: break
        if time.time() - t0 > timeout: raise TimeoutError(pid)
        time.sleep(2)
    mon.stop = True; secs = round(time.time() - t0, 1)
    entry = h[pid]; st = entry.get("status", {})
    if st.get("status_str") == "error":
        for msg in st.get("messages", []):
            if msg[0] == "execution_error": print(json.dumps(msg[1], indent=1)[:3000])
        raise RuntimeError("Error de ejecución (ver arriba)")
    files = []
    for out in entry.get("outputs", {}).values():
        for items in out.values():
            if not isinstance(items, list): continue
            for it in items:
                if isinstance(it, dict) and "filename" in it:
                    data = requests.get(f"{base}/view", params={"filename": it["filename"], "subfolder": it.get("subfolder", ""), "type": it.get("type", "output")}).content
                    p = f"{OUT_DIR}/{it['filename']}"; open(p, "wb").write(data); files.append(p)
    res = {"tag": tag, "seconds": secs, "peak_vram_mib": mon.peak, "files": files}
    print(json.dumps(res, indent=1)); return res

def check_image(path):
    a = np.asarray(Image.open(path).convert("RGB")).astype(np.float32)
    verdict = "FALLO: negra/plana (probable overflow fp16)" if a.mean() < 3 or a.std() < 2 else "OK"
    print(f"media {a.mean():.1f} | std {a.std():.1f} -> {verdict}")
    display(IPImage(path, width=512)); return verdict

def log(phase, params, res, verdict, note=""):
    row = {"ts": time.strftime("%Y-%m-%d %H:%M"), "fase": phase, "params": params,
           "seg": res.get("seconds"), "vram_pico_mib": res.get("peak_vram_mib"), "veredicto": verdict, "nota": note}
    open(LOG_PATH, "a").write(json.dumps(row, ensure_ascii=False) + "\n"); print("bitácora +1")

def multigpu_loader(std_node, inputs, device="cuda:1"):
    """Usa la variante MultiGPU del loader si existe (pone el componente en GPU 1); si no, el loader estándar."""
    mg = std_node + "MultiGPU"
    if has_node(mg):
        req = OBJECT_INFO[mg]["input"].get("required", {}); opt = OBJECT_INFO[mg]["input"].get("optional", {})
        if "device" in req or "device" in opt:
            return {"class_type": mg, "inputs": {**inputs, "device": device}}
    print(f"aviso: {mg} no disponible, se usa {std_node} en GPU 0")
    return {"class_type": std_node, "inputs": inputs}

print("Nodos clave:", {n: has_node(n) for n in ["UnetLoaderGGUF", "CLIPLoaderMultiGPU", "VAELoaderMultiGPU",
                                                 "WanImageToVideo", "CreateVideo", "SaveVideo", "VHS_VideoCombine", "ModelSamplingAuraFlow"]})

## Fase 2 — Imagen: Z-Image Turbo
Criterio: imagen **no negra** y tiempo < ~90 s a 1024². Si sale negra/plana → Fase 2B.

In [ ]:
ZI = {
    "unet": fetch([("Comfy-Org/z_image_turbo", "*diffusion_models/z_image_turbo_bf16*.safetensors"),
                   ("Comfy-Org/z_image_turbo", "*diffusion_models/*.safetensors")], "diffusion_models"),
    "clip": fetch([("Comfy-Org/z_image_turbo", "*text_encoders/qwen_3_4b*.safetensors")], "text_encoders"),
    "vae":  fetch([("Comfy-Org/z_image_turbo", "*vae/ae.safetensors"),
                   ("Comfy-Org/z_image_turbo", "*vae/*.safetensors")], "vae"),
}
print(ZI)
OBJECT_INFO = requests.get(f"{BASE}/object_info").json()  # refresca listas de modelos

In [ ]:
def wf_zimage(prompt, w=1024, h=1024, steps=8, seed=42, shift=3.0, te_on_gpu1=True):
    clip = multigpu_loader("CLIPLoader", {"clip_name": ZI["clip"], "type": "lumina2"}) if te_on_gpu1 \
           else {"class_type": "CLIPLoader", "inputs": {"clip_name": ZI["clip"], "type": "lumina2"}}
    return {
        "1": {"class_type": "UNETLoader", "inputs": {"unet_name": ZI["unet"], "weight_dtype": "default"}},
        "2": clip,
        "3": {"class_type": "VAELoader", "inputs": {"vae_name": ZI["vae"]}},
        "4": {"class_type": "ModelSamplingAuraFlow", "inputs": {"model": ["1", 0], "shift": shift}},
        "5": {"class_type": "CLIPTextEncode", "inputs": {"clip": ["2", 0], "text": prompt}},
        "6": {"class_type": "ConditioningZeroOut", "inputs": {"conditioning": ["5", 0]}},
        "7": {"class_type": "EmptySD3LatentImage", "inputs": {"width": w, "height": h, "batch_size": 1}},
        "8": {"class_type": "KSampler", "inputs": {"model": ["4", 0], "positive": ["5", 0], "negative": ["6", 0],
              "latent_image": ["7", 0], "seed": seed, "steps": steps, "cfg": 1.0,
              "sampler_name": "res_multistep", "scheduler": "simple", "denoise": 1.0}},
        "9": {"class_type": "VAEDecode", "inputs": {"samples": ["8", 0], "vae": ["3", 0]}},
        "10": {"class_type": "SaveImage", "inputs": {"images": ["9", 0], "filename_prefix": "zimage"}},
    }

PROMPT_IMG = ("Photorealistic portrait of a middle-aged fisherman on a wooden dock at golden hour, "
              "85mm lens, shallow depth of field, warm rim light from the low sun, detailed skin texture, "
              "calm sea and soft bokeh in the background")

# 1a ejecución incluye carga de modelos; la 2a mide la inferencia real.
for i in range(2):
    res = run_workflow(wf_zimage(PROMPT_IMG, seed=42 + i), tag=f"zimage_1024_run{i}")
    v = check_image(res["files"][0])
log("2-zimage", {"res": "1024x1024", "steps": 8}, res, v)

### Fase 2B — Plan B: SDXL + Lightning (ejecutar solo si la Fase 2 falló)
Incluye el VAE *fp16-fix*: el VAE original de SDXL produce NaN en fp16.

In [ ]:
RUN_2B = False
if RUN_2B:
    free_memory()
    SX = {
        "ckpt": fetch([("stabilityai/stable-diffusion-xl-base-1.0", "sd_xl_base_1.0.safetensors")], "checkpoints"),
        "vae":  fetch([("madebyollin/sdxl-vae-fp16-fix", "sdxl.vae.safetensors")], "vae"),
        "lora": fetch([("ByteDance/SDXL-Lightning", "sdxl_lightning_8step_lora.safetensors")], "loras"),
    }
    OBJECT_INFO = requests.get(f"{BASE}/object_info").json()
    wf = {
        "1": {"class_type": "CheckpointLoaderSimple", "inputs": {"ckpt_name": SX["ckpt"]}},
        "2": {"class_type": "LoraLoader", "inputs": {"model": ["1", 0], "clip": ["1", 1], "lora_name": SX["lora"], "strength_model": 1.0, "strength_clip": 1.0}},
        "3": {"class_type": "VAELoader", "inputs": {"vae_name": SX["vae"]}},
        "4": {"class_type": "CLIPTextEncode", "inputs": {"clip": ["2", 1], "text": PROMPT_IMG}},
        "5": {"class_type": "CLIPTextEncode", "inputs": {"clip": ["2", 1], "text": ""}},
        "6": {"class_type": "EmptyLatentImage", "inputs": {"width": 1024, "height": 1024, "batch_size": 1}},
        "7": {"class_type": "KSampler", "inputs": {"model": ["2", 0], "positive": ["4", 0], "negative": ["5", 0], "latent_image": ["6", 0],
              "seed": 42, "steps": 8, "cfg": 1.0, "sampler_name": "euler", "scheduler": "sgm_uniform", "denoise": 1.0}},
        "8": {"class_type": "VAEDecode", "inputs": {"samples": ["7", 0], "vae": ["3", 0]}},
        "9": {"class_type": "SaveImage", "inputs": {"images": ["8", 0], "filename_prefix": "sdxl"}},
    }
    res = run_workflow(wf, tag="sdxl_lightning_1024"); v = check_image(res["files"][0])
    log("2B-sdxl", {"res": "1024x1024", "steps": 8}, res, v)

### Fase 2C — Imagen inicial 16:9 para video
Wan genera a 832×480 y recorta por el centro: la imagen inicial debe ser horizontal. Medido: 28 s a 1280×720.

In [ ]:
PROMPT_START = ("Photorealistic waist-up portrait of a weathered 50-year-old fisherman wearing a faded navy work jacket "
                "and a dark knit cap, coiled rope over one shoulder, standing on a wooden dock at golden hour. "
                "85mm lens at f/1.8, eye-level camera, subject on the right third with open space on the left, "
                "shallow depth of field: the sea and moored fishing boats dissolve into creamy bokeh. "
                "Low sun behind camera-left casts a warm rim light on hair and shoulders, soft bounce fill on the face. "
                "Natural skin texture, fine wrinkles, light stubble, calm water with gentle specular highlights.")
res = run_workflow(wf_zimage(PROMPT_START, w=1280, h=720, seed=7), tag="zimage_1280x720_start")
v = check_image(res["files"][0])
log("2C-zimage-start", {"res": "1280x720", "steps": 8, "seed": 7}, res, v)

## Fase 3 — Video: Wan 2.2 I2V A14B (GGUF Q4) + lightx2v 4 pasos
Primero **33 frames** (calibración). Si pasa sin OOM, sube `LENGTH` a **81** (5 s @16 fps).
Antes de cargar Wan se libera la memoria de la Fase 2 (32 GB de RAM no alcanzan para ambos).

In [ ]:
free_memory()
WAN = {
    "high": fetch([("QuantStack/Wan2.2-I2V-A14B-GGUF", "*HighNoise*Q4_K_M.gguf")], "diffusion_models"),
    "low":  fetch([("QuantStack/Wan2.2-I2V-A14B-GGUF", "*LowNoise*Q4_K_M.gguf")], "diffusion_models"),
    # fp16 preferido (GPU1 tiene espacio y evita conversiones fp8 en Turing); fp8 como respaldo
    "te":   fetch([("Comfy-Org/Wan_2.1_ComfyUI_repackaged", "*text_encoders/umt5_xxl_fp16.safetensors"),
                   ("Comfy-Org/Wan_2.2_ComfyUI_Repackaged", "*text_encoders/umt5_xxl*.safetensors")], "text_encoders"),
    "vae":  fetch([("Comfy-Org/Wan_2.2_ComfyUI_Repackaged", "*vae/wan_2.1_vae.safetensors"),
                   ("Comfy-Org/Wan_2.1_ComfyUI_repackaged", "*vae/wan_2.1_vae.safetensors")], "vae"),
    "lora_high": fetch([("Comfy-Org/Wan_2.2_ComfyUI_Repackaged", "*loras/*i2v*lightx2v*4steps*high_noise*.safetensors")], "loras"),
    "lora_low":  fetch([("Comfy-Org/Wan_2.2_ComfyUI_Repackaged", "*loras/*i2v*lightx2v*4steps*low_noise*.safetensors")], "loras"),
}
print(json.dumps(WAN, indent=1))
OBJECT_INFO = requests.get(f"{BASE}/object_info").json()

In [ ]:
def wf_wan_i2v(image_name, prompt, neg="", w=832, h=480, length=33, seed=42, shift=8.0, fps=16):
    wf = {
        "1": {"class_type": "UnetLoaderGGUF", "inputs": {"unet_name": WAN["high"]}},
        "2": {"class_type": "UnetLoaderGGUF", "inputs": {"unet_name": WAN["low"]}},
        "3": {"class_type": "LoraLoaderModelOnly", "inputs": {"model": ["1", 0], "lora_name": WAN["lora_high"], "strength_model": 1.0}},
        "4": {"class_type": "LoraLoaderModelOnly", "inputs": {"model": ["2", 0], "lora_name": WAN["lora_low"], "strength_model": 1.0}},
        "5": {"class_type": "ModelSamplingSD3", "inputs": {"model": ["3", 0], "shift": shift}},
        "6": {"class_type": "ModelSamplingSD3", "inputs": {"model": ["4", 0], "shift": shift}},
        "7": multigpu_loader("CLIPLoader", {"clip_name": WAN["te"], "type": "wan"}),
        "8": multigpu_loader("VAELoader", {"vae_name": WAN["vae"]}),
        "9": {"class_type": "CLIPTextEncode", "inputs": {"clip": ["7", 0], "text": prompt}},
        "10": {"class_type": "CLIPTextEncode", "inputs": {"clip": ["7", 0], "text": neg}},
        "11": {"class_type": "LoadImage", "inputs": {"image": image_name}},
        "12": {"class_type": "WanImageToVideo", "inputs": {"positive": ["9", 0], "negative": ["10", 0], "vae": ["8", 0],
               "start_image": ["11", 0], "width": w, "height": h, "length": length, "batch_size": 1}},
        "13": {"class_type": "KSamplerAdvanced", "inputs": {"model": ["5", 0], "add_noise": "enable", "noise_seed": seed,
               "steps": 4, "cfg": 1.0, "sampler_name": "euler", "scheduler": "simple", "positive": ["12", 0], "negative": ["12", 1],
               "latent_image": ["12", 2], "start_at_step": 0, "end_at_step": 2, "return_with_leftover_noise": "enable"}},
        "14": {"class_type": "KSamplerAdvanced", "inputs": {"model": ["6", 0], "add_noise": "disable", "noise_seed": seed,
               "steps": 4, "cfg": 1.0, "sampler_name": "euler", "scheduler": "simple", "positive": ["12", 0], "negative": ["12", 1],
               "latent_image": ["13", 0], "start_at_step": 2, "end_at_step": 10000, "return_with_leftover_noise": "disable"}},
        "15": {"class_type": "VAEDecode", "inputs": {"samples": ["14", 0], "vae": ["8", 0]}},
    }
    if has_node("CreateVideo") and has_node("SaveVideo"):
        wf["16"] = {"class_type": "CreateVideo", "inputs": {"images": ["15", 0], "fps": float(fps)}}
        wf["17"] = {"class_type": "SaveVideo", "inputs": {"video": ["16", 0], "filename_prefix": "wan_i2v", "format": "auto", "codec": "auto"}}
    else:
        wf["16"] = {"class_type": "VHS_VideoCombine", "inputs": {"images": ["15", 0], "frame_rate": fps, "loop_count": 0,
                    "filename_prefix": "wan_i2v", "format": "video/h264-mp4", "pingpong": False, "save_output": True}}
    return wf

In [ ]:
# 3.1 Imagen inicial: la generas con Z-Image (pipeline T2I -> I2V) o subes la tuya a /kaggle/working
START_IMAGE = None   # p.ej. "/kaggle/working/mi_foto.png"; None = usa la última imagen de la Fase 2
if START_IMAGE is None:
    START_IMAGE = sorted([f"{OUT_DIR}/{f}" for f in os.listdir(OUT_DIR) if f.startswith(("zimage", "sdxl")) and f.endswith(".png")])[-1]
print("Imagen inicial:", START_IMAGE)
img_name = upload_image(START_IMAGE)

PROMPT_VID = "The fisherman slowly turns his head toward the sea, gentle breeze moving his hair, soft waves, warm golden light, subtle handheld camera"
LENGTH = 81   # 81 = 5 s (medido: ~12.4 min). 33 = 2 s para pruebas rápidas (~2.5 min en caliente)

res = run_workflow(wf_wan_i2v(img_name, PROMPT_VID, length=LENGTH), tag=f"wan_i2v_832x480_{LENGTH}f")
vid = [f for f in res["files"] if f.endswith((".mp4", ".webm"))]
if vid: display(Video(vid[0], embed=True, width=640))
log("3-wan-i2v", {"res": "832x480", "frames": LENGTH, "steps": "2+2", "quant": "Q4_K_M"}, res,
    "OK" if vid else "sin video", note="revisa visualmente: movimiento, coherencia, artefactos")

### Diagnóstico de rendimiento
Muestra si el modelo cargó completo o parcial y el tiempo por paso de muestreo.

In [ ]:
print(sh(r"tr '\r' '\n' < /kaggle/working/comfy_8188.log | grep -iE 'loaded (completely|partially)|Requested to load|Prompt executed|s/it|it/s' | tail -40"))

## Fase 4 — Exportar workflows para n8n
Guarda los workflows en formato API. n8n los reutiliza cambiando solo prompt/seed/imagen.

In [ ]:
os.makedirs("/kaggle/working/workflows", exist_ok=True)
json.dump(wf_zimage("{{PROMPT}}"), open("/kaggle/working/workflows/zimage_t2i.json", "w"), indent=1)
if "WAN" in globals():
    json.dump(wf_wan_i2v("{{IMAGE}}", "{{PROMPT}}", length=81), open("/kaggle/working/workflows/wan_i2v_81f.json", "w"), indent=1)
print(sh("ls -la /kaggle/working/workflows"))
print("""
Llamadas que hará n8n (HTTP Request):
  1) POST  {HOST}/upload/image           multipart 'image'          -> {name}
  2) POST  {HOST}/prompt                 {"prompt": <workflow JSON>} -> {prompt_id}
  3) GET   {HOST}/history/{prompt_id}    repetir hasta que aparezca  -> outputs
  4) GET   {HOST}/view?filename=..&subfolder=..&type=output          -> archivo
Con Cloudflare Access: añadir headers CF-Access-Client-Id y CF-Access-Client-Secret.
""")

### 4B — Probar la API desde fuera (a través del túnel)
Hasta ahora el notebook habló con ComfyUI por dentro (127.0.0.1). Esta celda usa la **URL pública** del túnel, que es exactamente lo que hará n8n.

In [ ]:
import re
# v5: añade las cabeceras de Cloudflare Access a toda petición que vaya a la URL pública
_orig_request = requests.sessions.Session.request
def _request_with_access(self, method, url, **kw):
    if ACCESS_HEADERS and str(url).startswith(PUBLIC):
        kw["headers"] = {**ACCESS_HEADERS, **(kw.get("headers") or {})}
    return _orig_request(self, method, url, **kw)
PUBLIC = (re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/kaggle/working/cloudflared.log").read()) or [None])[0]
PUBLIC = PUBLIC or "https://TU-HOSTNAME"   # con túnel con nombre, pon aquí tu hostname
requests.sessions.Session.request = _request_with_access
print("Probando:", PUBLIC)
print("system_stats:", requests.get(f"{PUBLIC}/system_stats", timeout=30).status_code)
res = run_workflow(wf_zimage("A red fishing boat at dawn, photorealistic", w=768, h=432, seed=1), base=PUBLIC, tag="api_publica")
check_image(res["files"][0])
log("4B-api-publica", {"via": "tunel", "res": "768x432"}, res, "OK")

## Bitácora — resumen para pegar en el documento

In [ ]:
rows = [json.loads(l) for l in open(LOG_PATH)] if os.path.exists(LOG_PATH) else []
print("| Fecha | Fase | Parámetros | Seg | VRAM pico MiB (GPU0/GPU1) | Veredicto | Nota |")
print("| --- | --- | --- | --- | --- | --- | --- |")
for r in rows:
    v = r["vram_pico_mib"] or {}
    print(f"| {r['ts']} | {r['fase']} | {json.dumps(r['params'], ensure_ascii=False)} | {r['seg']} | {v.get('0','?')}/{v.get('1','?')} | {r['veredicto']} | {r['nota']} |")

## Fase 5 — Modo servidor (v5)
Deja el notebook vivo para usar la API desde tu PC. **Ejecútala la última** y no la interrumpas: mientras corre, ComfyUI y el túnel se vigilan cada 60 s y se reinician si caen.
Con *Save & Run All* esta celda mantiene viva la sesión hasta `SERVER_HOURS`. Descarga los workflows de `/kaggle/working/workflows/` (Fase 4) para el cliente de PC.

In [ ]:
import datetime
def tunnel_alive(): return "cloudflared" in sh("pgrep -a cloudflared || true")
def comfy_alive():
    try: return requests.get(f"http://127.0.0.1:{PORT_A}/system_stats", timeout=10).status_code == 200
    except Exception: return False

def restart_tunnel():
    if not ENABLE_TUNNEL: return
    lf = open("/kaggle/working/cloudflared.log", "a")
    if CF_TUNNEL_TOKEN:
        subprocess.Popen([CF, "tunnel", "--no-autoupdate", "run", "--token", CF_TUNNEL_TOKEN], stdout=lf, stderr=subprocess.STDOUT)
    else:
        subprocess.Popen([CF, "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT_A}"], stdout=lf, stderr=subprocess.STDOUT)

if SERVER_MODE:
    t_end = time.time() + SERVER_HOURS * 3600
    n = 0
    print("Servidor activo. Interrumpe la celda para detenerlo.")
    try:
        while time.time() < t_end:
            ok_c, ok_t = comfy_alive(), (tunnel_alive() or not ENABLE_TUNNEL)
            if not ok_c:
                print(datetime.datetime.now(), "ComfyUI caído -> reiniciando"); start_comfy(PORT_A)
            if not ok_t:
                print(datetime.datetime.now(), "cloudflared caído -> reiniciando"); restart_tunnel()
            if n % 10 == 0:
                print(datetime.datetime.now().strftime("%H:%M"), "| ComfyUI:", ok_c, "| túnel:", ok_t)
            n += 1; time.sleep(60)
        print("SERVER_HOURS alcanzado: saliendo.")
    except KeyboardInterrupt:
        print("Servidor detenido a mano.")
else:
    print("SERVER_MODE=False: nada que hacer.")
